# 🛡️ SMS Spam Detection using Machine Learning & NLP
### Comprehensive Exploratory Data Analysis, Model Comparison & Pipeline

This notebook demonstrates the complete end-to-end Machine Learning lifecycle:
1. **Data Inspection & Cleaning** (Handling duplicates and nulls)
2. **Exploratory Data Analysis** (Class distribution, character length, word counts)
3. **Text Preprocessing** (Normalization, URL/phone replacement, regex cleaning)
4. **Stratified Train / Test Split** (Preventing class imbalance bias)
5. **TF-IDF Feature Extraction** (Fitting ONLY on training data to prevent data leakage)
6. **Model Training & Comparison** (Naive Bayes, Logistic Regression, Linear SVM)
7. **Evaluation Metrics** (Accuracy, Precision, Recall, F1-Score, Confusion Matrix)
8. **Model Serialization** (Saving model and vectorizer with joblib)

## 1. Import Essential Libraries

In [ ]:
import os
import re
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

sns.set_theme(style='whitegrid')
print('All libraries imported successfully!')

## 2. Load the UCI SMS Spam Collection Dataset

In [ ]:
data_path = os.path.join('..', 'data', 'SMSSpamCollection')
df = pd.read_csv(
    data_path,
    sep='\t',
    header=None,
    names=['label', 'message'],
    encoding='latin-1'
)
print(f'Raw dataset dimensions: {df.shape[0]} rows x {df.shape[1]} columns')
df.head()

## 3. Data Cleaning & Duplicate Handling
We inspect missing values and duplicate rows. Duplicate messages could skew cross-validation or leak between train and test splits, so we deduplicate them.

In [ ]:
print('Missing Values:\n', df.isnull().sum())
print('\nDuplicate Count:', df.duplicated().sum())

# Drop duplicates
df = df.drop_duplicates(keep='first').reset_index(drop=True)
print(f'Clean unique rows: {len(df)}')
print('\nClass Distribution:\n', df['label'].value_counts())

## 4. Exploratory Data Analysis (EDA)
Let's engineer message length features (characters and words) and compare spam vs. legitimate (ham) messages.

In [ ]:
df['char_count'] = df['message'].apply(len)
df['word_count'] = df['message'].apply(lambda msg: len(msg.split()))
df.groupby('label')[['char_count', 'word_count']].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(ax=axes[0], data=df, x='label', palette=['#10B981', '#EF4444'])
axes[0].set_title('Target Class Balance (Ham vs Spam)')

sns.histplot(ax=axes[1], data=df, x='char_count', hue='label', bins=50, kde=True, palette=['#10B981', '#EF4444'])
axes[1].set_title('Character Length: Spam vs Ham')
axes[1].set_xlim(0, 300)
plt.tight_layout()
plt.show()

## 5. Text Preprocessing
We normalize text to remove noise: lowercasing, replacing URLs with a token, replacing numbers with a token, and stripping special characters.

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', ' url ', text)
    text = re.sub(r'\b\d{5,}\b', ' num ', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_message'] = df['message'].apply(clean_text)
df['target'] = df['label'].map({'ham': 0, 'spam': 1})
df[['message', 'clean_message', 'target']].head()

## 6. Stratified Train / Test Split
We split 80% for training and 20% for testing using `stratify=y` to preserve the 87:13 ham:spam ratio in both subsets.

In [ ]:
X = df['clean_message']
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Train samples: {len(X_train)} | Test samples: {len(X_test)}')

## 7. TF-IDF Feature Extraction
**IMPORTANT RULE:** Fit ONLY on `X_train`, then transform `X_train` and `X_test`.

In [ ]:
vectorizer = TfidfVectorizer(stop_words='english', lowercase=True, max_features=3000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)
print(f'Vocabulary Size: {len(vectorizer.get_feature_names_out())}')
print(f'Train TF-IDF shape: {X_train_tfidf.shape}')
print(f'Test TF-IDF shape:  {X_test_tfidf.shape}')

## 8. Train Multiple ML Classifiers and Evaluate

In [ ]:
models = {
    'Multinomial Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Linear SVM (Calibrated)': CalibratedClassifierCV(LinearSVC(random_state=42, dual=False))
}

metrics_list = []
for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    cm = confusion_matrix(y_test, y_pred)
    metrics_list.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'False Positives (Legit marked Spam)': cm[0, 1],
        'False Negatives (Spam missed)': cm[1, 0]
    })

comparison_df = pd.DataFrame(metrics_list)
comparison_df

## 9. Save Best Model and Vectorizer

In [ ]:
os.makedirs(os.path.join('..', 'models'), exist_ok=True)
selected_model = models['Linear SVM (Calibrated)']
joblib.dump(selected_model, os.path.join('..', 'models', 'spam_model.pkl'))
joblib.dump(vectorizer, os.path.join('..', 'models', 'tfidf_vectorizer.pkl'))
print('Artifacts successfully saved to models/ directory!')